In [11]:
pip install sentence-transformers faiss-cpu rank-bm25 transformers torch tqdm

Defaulting to user installation because normal site-packages is not writeable
INFO: pip is looking at multiple versions of scipy to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 10.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 1.1 MB/s  0:00:04 eta 0:00:010m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.3/30.3 MB 22.6 MB/s  0:00:01 eta 0:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 1.22.1
    Uninstalling numpy-1.22.1:
      Successfully uninstalled numpy-1.22.1
  Attempting uninstall: scipy━━━━━━━━━━━━━━━━━━━ 0/5 [numpy]
    Found existing installation: scipy 1.7.3 0/5 [numpy]
    Uninstalling scipy-1.7.3:━━━━━━━━━━━━━━━ 0/5 [numpy]
      Successfully uninstalled scipy-1.7.3━━━━━━━━━━━━━━━━━━━━━━━━ 1/5 [scipy]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [sentence-transformers]ence-transformers]
ERROR: pip's dependency 

In [ ]:
import os
import re
import glob
import json
from typing import List
from tqdm import tqdm

# CONFIG
DOC_SOURCES = {
    "transformers": "/Users/zapashniysergey/Downloads/vs code docs /VKR_HSE/transformers/docs/source/en",
    "datasets": "/Users/zapashniysergey/Downloads/vs code docs /VKR_HSE/datasets/docs/source",
    "diffusers": "/Users/zapashniysergey/Downloads/vs code docs /VKR_HSE/diffusers/docs/source/en",
}

OUTPUT_DIR = "rag_corpus"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "corpus.jsonl")

CHUNK_SIZE = 400
CHUNK_OVERLAP = 50

os.makedirs(OUTPUT_DIR, exist_ok=True)

# TEXT CLEANING (SAFE)
def clean_markdown(text: str) -> str:
    text = re.sub(r"```.*?```", "", text, flags=re.S)
    text = re.sub(r"`[^`]*`", "", text)
    text = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", text)
    text = re.sub(r"#+", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def chunk_text(text: str, size: int, overlap: int) -> List[str]:
    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = start + size
        chunk_words = words[start:end]

        if len(chunk_words) >= 50:
            chunks.append(" ".join(chunk_words))

        start += size - overlap

    return chunks


# BUILD CORPUS
total_chunks = 0

with open(OUTPUT_FILE, "w", encoding="utf-8") as out:
    for library, base_path in DOC_SOURCES.items():
        print(f"\nProcessing {library} documentation...")

        md_files = glob.glob(
            os.path.join(base_path, "**", "*.md"),
            recursive=True
        )

        print(f"Found {len(md_files)} markdown files")

        for file_path in tqdm(md_files):
            try:
                with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                    raw_text = f.read()
            except Exception:
                continue

            cleaned_text = clean_markdown(raw_text)
            chunks = chunk_text(cleaned_text, CHUNK_SIZE, CHUNK_OVERLAP)

            for chunk in chunks:
                record = {
                    "text": chunk,
                    "source": os.path.relpath(file_path, base_path),
                    "library": library
                }
                out.write(json.dumps(record, ensure_ascii=False) + "\n")
                total_chunks += 1

print("\nCorpus built successfully!")
print(f"Total chunks: {total_chunks}")

with open(os.path.join(OUTPUT_DIR, "stats.txt"), "w", encoding="utf-8") as f:
    f.write(f"Total chunks: {total_chunks}\n")


Processing transformers documentation...
Found 598 markdown files


100%|██████████| 598/598 [00:00<00:00, 2311.31it/s]



Processing datasets documentation...
Found 4 markdown files


100%|██████████| 4/4 [00:00<00:00, 1776.31it/s]



Processing diffusers documentation...
Found 359 markdown files


100%|██████████| 359/359 [00:00<00:00, 3114.26it/s]


Corpus built successfully!
Total chunks: 1884


In [ ]:
import json
import numpy as np
from tqdm import tqdm

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import faiss


# LOAD CORPUS

CORPUS_PATH = "rag_corpus/corpus.jsonl"

documents = []
metadata = []

with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        documents.append(record["text"])
        metadata.append({
            "source": record["source"],
            "library": record["library"]
        })

print(f"Loaded {len(documents)} documents")


# BM25 RETRIEVAL

tokenized_docs = [doc.lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized_docs)


def bm25_retrieve(query: str, k: int = 5):
    scores = bm25.get_scores(query.lower().split())
    top_idx = np.argsort(scores)[::-1][:k]
    return [
        {
            "text": documents[i],
            "score": scores[i],
            **metadata[i]
        }
        for i in top_idx
    ]


# DENSE RETRIEVAL (SBERT + FAISS)

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embedder = SentenceTransformer(EMBEDDING_MODEL)

print("Encoding documents...")
doc_embeddings = embedder.encode(
    documents,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

dim = doc_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(doc_embeddings)

print("FAISS index built")


def dense_retrieve(query: str, k: int = 5):
    query_emb = embedder.encode(
        [query],
        normalize_embeddings=True
    )
    scores, idx = index.search(query_emb, k)

    return [
        {
            "text": documents[i],
            "score": float(scores[0][j]),
            **metadata[i]
        }
        for j, i in enumerate(idx[0])
    ]

# TEST INTERFACE

print("\nRetrieval system ready. Type 'exit' to stop.\n")

while True:
    query = input("Query: ")
    if query.lower() == "exit":
        break

    print("\n--- BM25 RESULTS ---")
    bm25_results = bm25_retrieve(query, k=3)
    for r in bm25_results:
        print(f"[{r['library']}] {r['text'][:200]}...\n")

    print("\n--- DENSE RESULTS ---")
    dense_results = dense_retrieve(query, k=3)
    for r in dense_results:
        print(f"[{r['library']}] {r['text'][:200]}...\n")

    print("=" * 80)

Loaded 1884 documents
Encoding documents...


Batches:   0%|          | 0/30 [00:00<?, ?it/s]

FAISS index built

Retrieval system ready. Type 'exit' to stop.



In [ ]:
import random
import re
import numpy as np
from tqdm import tqdm



# QUERY GENERATION
def generate_query_from_chunk(text: str) -> str:
    """
    Take first 1–2 sentences from a chunk as a pseudo-query.
    """
    sentences = re.split(r"[.!?]", text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 20]
    return " ".join(sentences[:2])

# EVALUATION
def evaluate_retriever(
    retrieve_fn,
    documents,
    metadata,
    k=5,
    n_samples=200,
    seed=42
):
    random.seed(seed)
    indices = random.sample(range(len(documents)), n_samples)

    hits = 0
    rr_sum = 0.0

    for idx in tqdm(indices, desc="Evaluating"):
        query = generate_query_from_chunk(documents[idx])
        true_source = metadata[idx]["source"]

        results = retrieve_fn(query, k=k)

        found = False
        for rank, res in enumerate(results, start=1):
            if res["source"] == true_source:
                hits += 1
                rr_sum += 1.0 / rank
                found = True
                break

    recall = hits / n_samples
    mrr = rr_sum / n_samples

    return {
        f"Recall@{k}": round(recall, 4),
        "MRR": round(mrr, 4),
        "Samples": n_samples
    }



# RUN EVALUATION
print("\nRunning BM25 evaluation...")
bm25_metrics = evaluate_retriever(
    bm25_retrieve,
    documents,
    metadata,
    k=5,
    n_samples=200
)

print("\nRunning Dense evaluation...")
dense_metrics = evaluate_retriever(
    dense_retrieve,
    documents,
    metadata,
    k=5,
    n_samples=200
)

print("\n=== RESULTS ===")
print("BM25 :", bm25_metrics)
print("Dense:", dense_metrics)


Running BM25 evaluation...


Evaluating: 100%|██████████| 200/200 [00:01<00:00, 182.78it/s]



Running Dense evaluation...


Evaluating: 100%|██████████| 200/200 [00:03<00:00, 63.99it/s]


=== RESULTS ===
BM25 : {'Recall@5': 0.515, 'MRR': 0.4921, 'Samples': 200}
Dense: {'Recall@5': 0.465, 'MRR': 0.4341, 'Samples': 200}


```
В наших экспериментах BM25 превосходит плотный поиск в протоколе оценки, учитывающем источник. Это ожидаемо, поскольку запросы формируются непосредственно из текста документа и имеют высокое лексическое совпадение с исходными документами. Плотный поиск менее эффективен в этих условиях, но остается выгодным для более абстрактных запросов пользователей.
```

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# LOAD LLM (simple baseline)

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

def generate_answer(context: str, question: str) -> str:
    prompt = f"""
You are an assistant answering questions based ONLY on the provided documentation context.
If the answer is not contained in the context, say "I don't know".

Context:
{context}

Question:
{question}

Answer:
"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.2
        )

    return tokenizer.decode(output[0], skip_special_tokens=True).split("Answer:")[-1].strip()


# REAL USER QUERIES

TEST_QUERIES = [
    "What is the Trainer class used for in transformers?",
    "How can I fine-tune a pretrained transformer model?",
    "How do I load a dataset from the Hugging Face Hub?",
    "What is the difference between AutoModel and AutoModelForSequenceClassification?",
    "How are diffusion pipelines used in diffusers?"
]

TOP_K = 5

# RUN RAG DEMO

for query in TEST_QUERIES:
    print("=" * 100)
    print(f"QUERY:\n{query}\n")

    retrieved = dense_retrieve(query, k=TOP_K)

    print("RETRIEVED CONTEXT (top chunks):\n")
    for i, r in enumerate(retrieved, 1):
        print(f"[{i}] ({r['library']} | {r['source']})")
        print(r["text"][:300])
        print()

    context = "\n\n".join([r["text"] for r in retrieved])

    answer = generate_answer(context, query)

    print("MODEL ANSWER:\n")
    print(answer)
    print("\n")

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk.


QUERY:
What is the Trainer class used for in transformers?



Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


RETRIEVED CONTEXT (top chunks):

[1] (transformers | custom_models.md)
the Hub. The pretrained weights, configuration, and files should all be uploaded to the Hub now in a repository under your namespace. Because a custom model doesn't use the same modeling code as a Transformers' model, you need to add in [] to load it. Refer to the load custom models section for more

[2] (diffusers | api/models/transformer_bria_fibo.md)
<!--Copyright 2025 The HuggingFace Team. All rights reserved. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applica

[3] (diffusers | api/modular_diffusers/guiders.md)
Guiders Guiders are components in Modular Diffusers that control how the diffusion process is guided during generation. They implement various guidance techniques to improve generation quality and control. BaseGuidance [

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


RETRIEVED CONTEXT (top chunks):

[1] (transformers | index.md)
Fast text generation with large language models (LLMs) and vision language models (VLMs), including support for streaming and multiple decoding strategies. Design > [!TIP] > Read our Philosophy to learn more about Transformers' design principles. Transformers is designed for developers and machine l

[2] (transformers | add_new_model.md)
lacking documentation or if the codebase is complex. But you should use this as your motivation to implement the model in Transformers. Your contribution makes it more accessible and user-friendly to everyone! Orient yourself with the original repository by doing the following. - Locate the pretrain

[3] (transformers | model_doc/bitnet.md)
sequence length of **4096 tokens**. * *Recommendation:* For optimal performance on tasks requiring very long contexts (beyond the pre-training length or for specialized long-reasoning tasks), we recommend performing intermediate long-sequence adaptation/

## Комментарий по результатам RAG-бейзлайна (qualitative evaluation)

### Наблюдение

По результатам выполнения end-to-end RAG-пайплайна видно, что система **формально работает**, но качество ответа на пользовательские запросы на данном этапе **неудовлетворительное**.

Пример для запроса:

> *What is the Trainer class used for in transformers?*

- Retrieval вернул **нерелевантные фрагменты**:
  - документы из `diffusers`
  - служебные файлы (`custom_models.md`, license headers)
  - отсутствие файла `trainer.md` или `training.md` в top-K
- Сгенерированный ответ:является **бессмысленным** и не основан на документации.

---

### Интерпретация результата

Данный результат **не означает, что RAG-бейзлайн не работает**. Он указывает на ожидаемые ограничения первого базового решения:

1. **Проблема retrieval, а не генерации**
 - LLM не получил релевантный контекст
 - модель корректно не стала «галлюцинировать», но не смогла ответить

2. **Отсутствие фильтрации по домену**
 - Retrieval не ограничен библиотекой `transformers`
 - В top-K попадают документы из `diffusers`, не относящиеся к запросу

3. **Нет re-ranking**
 - Dense retrieval без cross-encoder re-ranker
 - Шумные и служебные чанки вытесняют целевые

4. **Слишком общий retrieval-бейзлайн**
 - Используется универсальная embedding-модель
 - Нет адаптации под API-документацию и технические термины

---

### Почему это ожидаемо для бейзлайна

Такое поведение является **типичным для первого RAG-бейзлайна** и часто демонстрируется в исследованиях как отправная точка:

- Retrieval корректен с точки зрения инфраструктуры
- Generation корректно зависит от retrieved контекста
- Ошибки носят **системный, а не случайный характер**

Это означает, что система:
- пока не даёт качественные ответы  
- но уже **архитектурно корректна**

---

### Вывод

Полученный результат подтверждает:

- корректную реализацию пайплайна *query → retrieval → context → generation*
- необходимость дальнейших улучшений retrieval-части

Данный эксперимент используется как **baseline**, относительно которого в дальнейшей работе могут быть продемонстрированы улучшения, такие как:
- фильтрация по библиотеке
- hybrid retrieval (BM25 + Dense)
- cross-encoder re-ranking
- оптимизация контекста

---

### Статус этапа

**Baseline RAG: реализован и протестирован**  
**Качество: недостаточное (ожидаемо)**  
**Следующий шаг: улучшение retrieval и re-ranking**